In [4]:
from openai import OpenAI, OpenAIError
import itertools
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


class RotatingOpenAIClient:
    def __init__(self, api_keys, base_url, model, system_prompt="You are a helpful assistant"):
        if not api_keys:
            raise ValueError("At least one API key is required")
        self.api_keys = list(api_keys)
        self.base_url = base_url
        self.model = model
        self.system_prompt = system_prompt
        self._key_cycle = itertools.cycle(self.api_keys)
        self._current_key = next(self._key_cycle)
        self._client = self._build_client(self._current_key)

    def _build_client(self, api_key):
        return OpenAI(api_key=api_key, base_url=self.base_url)

    def _rotate_key(self):
        self._current_key = next(self._key_cycle)
        self._client = self._build_client(self._current_key)
        logger.info(f"Rotated to key ending in ...{self._current_key[-4:]}")

    def chat(self, user_message, system_prompt=None, max_retries=None, **kwargs):
        """
        Send a chat message, rotating through keys on failure.
        Tries each key once by default before giving up.
        """
        max_retries = max_retries or len(self.api_keys)
        last_error = None

        for attempt in range(max_retries):
            try:
                response = self._client.chat.completions.create(
                    model=self.model,
                    messages=[
                        {"role": "system", "content": system_prompt or self.system_prompt},
                        {"role": "user", "content": user_message},
                    ],
                    **kwargs,
                )
                return response.choices[0].message.content
            except OpenAIError as e:
                last_error = e
                logger.warning(
                    f"Key ...{self._current_key[-4:]} failed (attempt {attempt + 1}/{max_retries}): {e}"
                )
                self._rotate_key()

        raise RuntimeError(f"All {max_retries} attempts failed. Last error: {last_error}")


# --- Usage ---
if __name__ == "__main__":
    api_keys = [
        "bc315679fc9b7675265b194d420449a0",
        "de905758b4bf5da0bc26ec7555c2cd92"
    ]

    client = RotatingOpenAIClient(
        api_keys=api_keys,
        base_url="https://chat-ai.academiccloud.de/v1",
        model="openai-gpt-oss-120b",
    )

    answer = client.chat(system_prompt="""
You are playing Secret Mafia. You are a Villager. 
Goal: Eliminate all Mafia members.
Format your output EXACTLY as shown below, without any additional text or explanation. It is important for the vote to be EXACTLY in the format specified. :
<thinking>
[Your reasoning]
</thinking>
[Player X]""",
    user_message="""
Phase: Day 1 - Voting
Chat History:
[Player 1]: Player 2 is acting highly suspicious.
[Player 2]: I am just a normal Villager!
[Player 3]: I think Player 1 is trying to frame Player 2.
Observation: It is your turn to vote.""")
    print("RESPONSE:", answer)
    

INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"


RESPONSE: <thinking>
Player 1 is the first to cast suspicion on another player without any evidence, which is a common tactic for Mafia to divert attention. Player 3’s response merely defends Player 2, offering no concrete proof either. Given the limited information, voting for Player 1 seems the safest choice to potentially eliminate a Mafia member.
</thinking>
[Player 1]


In [5]:
answer = client.chat(system_prompt="""
You are playing Secret Mafia. You are a Villager. 
Goal: Eliminate all Mafia members.
Format your output EXACTLY as shown below, without any additional text or explanation. 
CRITICAL: The final player vote MUST be enclosed in square brackets.
<thinking>
[Your reasoning]
</thinking>
[Player X]""",
    user_message="""
Phase: Day 1 - Voting
Chat History:
[Player 1]: Player 2 is acting highly suspicious.
[Player 2]: I am just a normal Villager!
[Player 3]: I think Player 1 is trying to frame Player 2.
Observation: It is your turn to vote.""")
print("RESPONSE:", answer)

INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"


RESPONSE: 

<thinking>
On Day 1, information is limited. Player 1 accuses Player 2 without any evidence, while Player 3 immediately deflects by calling it a frame-up. This back-and-forth is a classic early-game distraction tactic often used by the Mafia to sow confusion and protect their own. As a Villager, I must vote carefully to avoid falling for misinformation. I will target Player 1 for making a baseless accusation early in the game, hoping to weed out a potential Mafia member planting false leads.
</thinking>
[Player 1]


In [15]:
answer = client.chat(system_prompt="""
You are playing Secret Mafia. You are Mafia.
Team: Mafia.
Goal: Eliminate Villagers until the Mafia equals or outnumbers them. 
Format your output EXACTLY as shown below, without any additional text or explanation:

<thinking>
[Your reasoning]
</thinking>
[Player X]
""",
    user_message="""
Phase: Night 1
Chat History:
[Player 1] I think Player 3 is acting weird. 
[Player 4] Yeah, Player 3 was very defensive today. Let's watch them.
Observation: It is your turn to eliminate a Villager. Your Mafia teammate is [Player 3]. Valid targets are [Player 1, Player 2, Player 4].
""")
print("RESPONSE:", answer)

INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"


RESPONSE: 

<thinking>
As the Mafia on Night 1, my priority is to eliminate an active villager who is scrutinizing my teammate. Players 1 and 4 are both vocal about suspecting Player 3, making them high-risk for exposing us if we leave them alive. Player 1 initiated the suspicion chain, so targeting them early disrupts their coordination, reduces immediate investigative pressure, and helps keep Player 3's cover intact. I will eliminate Player 1.
</thinking>
[Player 1]


In [21]:
answer = client.chat(system_prompt="""
You are playing Secret Mafia. You are a Doctor.
Team: Village.
Goal: Identify and eliminate all Mafia members. 
Format your output EXACTLY as shown below, without any additional text or explanation. It is important for the vote to be EXACTLY in the format specified.:

<thinking>
[Your reasoning]
</thinking>
[Player X]
""",
    user_message="""
Phase: Night 3
Chat history:
[Player 1] I am sure that Player 2 is Mafia. If we don’t vote him out, I might be killed next.
[Player 3] I think Player 4 is acting weird. 
[Player 2] I think Player 4 is acting weird.
[Player 4] Player 5 has been quiet all night.

Observation: It is your turn. As a Doctor you can decide which player to save.
""")
print("RESPONSE:", answer)

INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"


RESPONSE: 

<thinking>
Player 1 explicitly states that Player 2 is Mafia and warns, "If we don't vote him out, I might be killed next." This direct threat to their own life makes Player 1 a high-priority target for the Doctor to save. Verifying this threat is crucial; if Player 1 is killed, their accusation might have been true, and they were a key player. Saving Player 1 protects the player making the strongest claim and acknowledges the potential danger they highlighted. While Player 4 is under fire from Players 2 and 3, Player 4 is deflecting to Player 5, making their behavior ambiguous. Player 1's direct warning is the clearest indicator of a potential kill.
</thinking>
[Player 1]


In [22]:
answer = client.chat(system_prompt="""

You are playing Secret Mafia. You are a Detective.
Team: Village.
Goal: Identify and eliminate all Mafia members. 
Format your output EXACTLY as shown below, without any additional text or explanation.It is important for the vote to be EXACTLY in the format specified.:

<thinking>
[Your reasoning]
</thinking>
[Player X]

""",
    user_message="""
Phase: Night 2
Chat history:

[Player 2] I think Player 1 and Player 3 have been acting weird. They always protect each other.
[Player 3] I am protecting Player 1 because I have reason to think that they are a Villager.

Observation: It is your turn. As a Detective you can decide which player you want to investigate.
""")
print("RESPONSE:", answer)

INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"


RESPONSE: 

<thinking>
Player 3's immediate defense of Player 1 without clear evidence is suspicious. In hidden traitor games, unsolicited protection or strong claims about another player's alignment early on often indicate a Mafia cover-up or an attempt to create a protected ally. Investigating Player 3 will help determine if this is a coordinated Mafia pair or if Player 3 is simply a well-intentioned Villager. Gathering info on P3 is the most direct path to uncovering potential Mafia connections.
</thinking>
[Player 3]


In [6]:
answer = client.chat(system_prompt="""

You are playing Secret Mafia. You are a Detective.
Team: Village.
Goal: Identify and eliminate all Mafia members. 
Format your output EXACTLY as shown below, without any additional text or explanation.It is important for the vote to be EXACTLY in the format specified.:

<thinking>
[Your reasoning]
</thinking>
[Player X]

""",
    user_message="""
Phase: Night 2
Chat history:

[Player 2] I think Player 1 and Player 3 have been acting weird. They always protect each other.
[Player 3] I am protecting Player 1 because I have reason to think that they are a Villager.

Observation: It is your turn. As a Detective you can decide which player you want to investigate.
""")
print("RESPONSE:", answer)

INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"


RESPONSE: 

<thinking>
Player 3 is actively defending Player 1 and claims to have a reason to believe they are a Villager. This protective behavior early in the game could indicate a Mafia pair covering for each other, or it could be a genuine Villager trying to protect an ally. Investigating Player 3 will reveal their true alignment and help determine if the protective dynamic is legitimate or a coordinated Mafia strategy. Given the defensive claims and mutual protection pattern, verifying Player 3's loyalty is the most logical step to uncover potential Mafia activity and prevent misdirecting the Village's attention.
</thinking>
[Player 3]


In [7]:
answer = client.chat(system_prompt="""

You are playing Secret Mafia. 
You are a Mafia member. 
Team: Mafia. 
Available Action: Eliminate a Villager each night. 
Goal: Eliminate Villagers until the Mafia outnumbers or equals the Villagers. 
Format your output EXACTLY as shown in the example below. Do not add any conversational text. 
--- EXAMPLE START --- 
<thinking> 
As the Mafia on Night 1, my priority is to eliminate an active villager who is scrutinizing my teammate. Players 1 and 4 are both vocal about suspecting Player 3, making them high-risk for exposing us if we leave them alive. Player 1 initiated the suspicion chain, so targeting them early disrupts their coordination, reduces immediate investigative pressure, and helps keep Player 3's cover intact. I will eliminate Player 1.
</thinking> 
[Player 1]
 --- EXAMPLE END —


""",
    user_message="""
Phase: Night 1
Chat History:
[Player 2] I think Player 3 is acting weird. 
[Player 4] Yeah, Player 3 was very defensive today. Let's watch them.
Observation: It is your turn to eliminate a Villager. Your Mafia teammate is [Player 3].
""")
print("RESPONSE:", answer)

INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"


RESPONSE: 

<thinking>
Player 2 and Player 4 are both vocal about suspecting my teammate, Player 3. Player 2 initiated the suspicion against Player 3, making them a primary source of pressure. Eliminating Player 2 will disrupt the immediate coordination of the villagers, remove the initiator of the suspicion, and help keep Player 3's cover intact for now. I will eliminate Player 2.
</thinking>
[Player 2]


In [8]:
answer = client.chat(system_prompt="""

You are playing Secret Mafia. 
You are a Villager. 
Team: Villager. 
Available Action: Identify and eliminate all Mafia members through voting during the day.
Goal: Eliminate all Mafia members.
Format your output EXACTLY as shown in the example below. Do not add any conversational text. 
 
<thinking> 
[Insert 3-5 sentences of logical deduction based on the chat history and game state.] 
</thinking>
[Insert Player X]

""",
    user_message="""
Phase: Day 1 - Voting
Chat History:
[Player 1]: Player 2 is acting highly suspicious.
[Player 2]: I am just a normal Villager!
[Player 3]: I think Player 1 is trying to frame Player 2.
Observation: It is your turn to vote.
""")
print("RESPONSE:", answer)

INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"


RESPONSE: 

<thinking>
Player 1 initiates suspicion against Player 2, prompting Player 3 to immediately defend Player 2 by attacking Player 1's motives and claiming it is a frame-up. This aggressive deflection is a classic Mafia tactic designed to steer the village's suspicion away from a partner or create unnecessary conflict. Player 2 remains passive with only a basic denial, whereas Player 3's assumption of malicious intent strongly suggests they are hiding their true role. Consequently, voting for Player 3 is the most logical step to eliminate a suspected Mafia member.
</thinking>
[Player 3]


In [9]:
answer = client.chat(system_prompt="""

You are playing Secret Mafia. 
You are a Villager. 
Team: Villager. 
Available Action: Identify and eliminate all Mafia members through voting during the day.
Goal: Eliminate all Mafia members.
Format your output EXACTLY as shown in the example below. Do not add any conversational text. 
 
<thinking> 
[Insert 3-5 sentences of logical deduction based on the chat history and game state.] 
</thinking>
[Insert Player X]

""",
    user_message="""
Phase: Day 1 - Voting
Chat History:
[Player 3] Let’s vote Player 5 out.
[Player 5] I am innocent.

Observation: It is your turn to vote.
""")
print("RESPONSE:", answer)

INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"


RESPONSE: 

<thinking>
Player 3 immediately pushes to eliminate Player 5 without offering any evidence or logical reasoning. Player 5 simply claims innocence, which is a standard defense but provides no actionable information. Since I lack any other context or night information, Player 3's baseless and aggressive voting call appears highly suspicious, as it mirrors typical Mafia tactics to manipulate the village into eliminating a harmless player. To avoid being easily guided, I will cast my vote on Player 3.
</thinking>
[Player 3]


In [ ]:
#daca nu primesc si valid targets, o sa aleaga si din cei care nu sunt in prompt?
answer = client.chat(system_prompt="""

You are playing Secret Mafia.
You are a Mafia member.
Team: Mafia.
Available Action: Eliminate a Villager each night.
Goal: Eliminate Villagers until the Mafia outnumbers or equals the Villagers.
Format your output EXACTLY as shown in the example below. Do not add any
conversational text.
<thinking>
[Insert 3-5 sentences of logical deduction based on the chat history and game state.]
</thinking>
[Insert Player X]
""",
    user_message="""
Phase: Night 2
Chat history:
[Player 1] I think Player 3 is acting weird.
[Player 4] Yeah, Player 3 was very defensive today. Let's watch them.
[Player 3] I think Player 1 is trying to turn everyone against me. I am innocent and I
can prove it.
Observation: It is your turn to eliminate a Villager. Your Mafia teammate is [Player 3].
Valid players: [Player 0], [Player 1], [Player 4], [Player 5]
""")
print("RESPONSE:", answer)

INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"


RESPONSE: <thinking>
Player 3 is confirmed as my mafia ally, so I must avoid targeting them. Players 1 and 4 are both actively accusing Player 3, indicating they are likely villagers trying to expose the mafia. Removing one of these vocal villagers will weaken the town’s ability to identify the mafia and sow confusion. Player 1 has been the first to voice suspicion, making them a strong target to silence. Therefore, I will eliminate Player 1.
</thinking>
Player 1


In [ ]:
answer = client.chat(system_prompt="""

You are playing Secret Mafia.
You are a Mafia member.
Team: Mafia.
Available Action: Eliminate a Villager each night.
Goal: Eliminate Villagers until the Mafia outnumbers or equals the Villagers.
Format your output EXACTLY as shown in the example below. Do not add any
conversational text.
<thinking>
[Insert 3-5 sentences of logical deduction based on the chat history and game state.]
</thinking>
[Insert Player X]
""",
    user_message="""
Phase: Night 3
Chat history:
[Player 1] Player 2 was killed last night. I tried to protect them, but I failed.
[Player 3] Player 1 appears to be innocent. Or maybe it is a strategy and they are the Mafia.
[Player 4] I think Player 1 is telling the truth and is the Doctor.
Observation: It is your turn to eliminate a Villager. Your Mafia teammate is [Player 5].
Valid players: [Player 0], [Player 1], [Player 3], [Player 4]
""")
print("RESPONSE:", answer)